![Analytics](https://images.unsplash.com/photo-1551288049-bebda4e38f71?w=1200&h=220&fit=crop)

# Three Machine Learning Prototypes for DataVine Analytics

### Classification, recommendation, and clustering across three client sectors

## Overview

Bottom Line Up Front: We built three working prototypes for three different client problems. The k-NN wine classifier reaches 95.6% test accuracy after PCA and hyperparameter tuning. The feed recommendation system groups the six chicken feeds into a high-growth and a low-growth family. The crime clustering model splits the 50 US states into three interpretable risk tiers.

What we did in each project: for the winery client we compressed 13 chemical measurements into 10 principal components that keep 96% of the variance, then used GridSearchCV to find the best k value and distance metric. For the agricultural client we profiled each feed type and used cosine similarity on the reduced data to find substitutes. For the public safety client we selected the three crime features, reduced them to two components, and compared K-Means against a Gaussian Mixture Model.

One honest finding we want to flag early: the recommendation brief asks for cosine similarity on a single principal component, and that combination is mathematically degenerate. With one dimension every value is a scalar, so cosine similarity can only ever return +1 or -1, which tells us sign agreement and nothing about how close two feeds are. We implement it as specified, show why it collapses, and add a distance-based similarity on the same component that actually ranks feeds usefully. Section 4 explains this in full.

## Setup and Data Loading

We load all three datasets first so we can inspect them together.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

sns.set_style('whitegrid')
print('Libraries loaded')

In [ ]:
# Dataset 1: Wine
wine_df = pd.read_csv('wine.csv')

# The 'Wine' column holds the cultivar label (1, 2 or 3), everything else is a feature
wine_features = wine_df.drop(columns='Wine')
wine_target = wine_df['Wine']
wine_target_names = ['Cultivar 1', 'Cultivar 2', 'Cultivar 3']

print('Wine shape:', wine_df.shape)
print('Feature columns:', list(wine_features.columns))
wine_df.head()

In [ ]:
# Dataset 2: Chickwts
chick_df = pd.read_csv('chickwts.csv')

# The file carries an R row-number column we do not need
if 'rownames' in chick_df.columns:
    chick_df = chick_df.drop(columns='rownames')

print('Chickwts shape:', chick_df.shape)
print('Feed types:', sorted(chick_df['feed'].unique()))
chick_df.head()

In [ ]:
# Dataset 3: USArrests
arrests_df = pd.read_csv('USArrests.csv')
arrests_df = arrests_df.set_index('State')

print('USArrests shape:', arrests_df.shape)
arrests_df.head()

## 1. Dataset Preparation

Before modelling anything we check all three datasets for missing values and inconsistencies, then standardize the numeric features. Standardizing matters here because every technique we use in this notebook (k-NN, PCA, K-Means, GMM) is distance based, so a feature measured in hundreds would otherwise drown out one measured in single digits.

In [ ]:
# Check all three datasets for missing values
print('=== MISSING VALUES ===')
print('Wine:      ', wine_df.isnull().sum().sum())
print('Chickwts:  ', chick_df.isnull().sum().sum())
print('USArrests: ', arrests_df.isnull().sum().sum())

print()
print('=== DUPLICATE ROWS ===')
print('Wine:      ', wine_df.duplicated().sum())
print('Chickwts:  ', chick_df.duplicated().sum())
print('USArrests: ', arrests_df.duplicated().sum())

In [ ]:
# Check data types and value ranges for inconsistencies
print('=== WINE ===')
print(wine_features.describe().round(2).T[['min','max','mean']])
print()
print('=== CHICKWTS ===')
print(chick_df['weight'].describe().round(2))
print('Feed types:', sorted(chick_df['feed'].unique()))
print()
print('=== USARRESTS ===')
print(arrests_df.describe().round(2).T[['min','max','mean']])

What the checks show. None of the three datasets has missing values, so no imputation is needed. The value ranges are all physically sensible, with no negative crime rates or negative weights, so there are no obvious data entry errors to correct.

The duplicate check flags one row in chickwts, which turns out to be two different chicks that both weighed 248g on soybean feed. That is a genuine coincidence rather than a recording error, so we keep both rows. Removing them would throw away a real observation.

The one thing that does need fixing is scale. In USArrests, Assault runs from 45 to 337 while Murder runs from 0.8 to 17.4, so Assault would dominate any distance calculation by roughly twenty to one. Wine has the same problem, with proline in the hundreds and hue near 1. We handle this with StandardScaler in each project below, which converts every feature to a mean of 0 and a variance of 1.

In [ ]:
# Standardize the numeric features in each dataset
# We fit a separate scaler per dataset since they are independent projects

# Wine
wine_scaler = StandardScaler()
wine_scaled = wine_scaler.fit_transform(wine_features)

# Chickwts (only one numeric feature)
chick_scaler = StandardScaler()
chick_df['weight_scaled'] = chick_scaler.fit_transform(chick_df[['weight']])

# USArrests
arrests_scaler = StandardScaler()
arrests_scaled = arrests_scaler.fit_transform(arrests_df)

print('Wine scaled shape:     ', wine_scaled.shape)
print('Chickwts scaled shape: ', chick_df[['weight_scaled']].shape)
print('USArrests scaled shape:', arrests_scaled.shape)
print()
print('Check: wine scaled mean is about 0 and std about 1')
print('  mean:', round(wine_scaled.mean(), 6))
print('  std: ', round(wine_scaled.std(), 6))

In [ ]:
# Visual check that scaling worked on USArrests
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

arrests_df.boxplot(ax=axes[0])
axes[0].set_title('USArrests Before Scaling')
axes[0].set_ylabel('Original units')

pd.DataFrame(arrests_scaled, columns=arrests_df.columns).boxplot(ax=axes[1])
axes[1].set_title('USArrests After Scaling')
axes[1].set_ylabel('Standard deviations')

plt.tight_layout()
plt.show()

print('Before scaling Assault towers over the other features.')
print('After scaling all four sit on the same range, so none can dominate.')

## 2. Project One: k-NN Classification for the Winery Client

The client wants to identify which of three cultivars a wine came from using its chemical measurements. We have 13 chemical features, several of which are correlated, so we use PCA to compress them before classifying. Then we tune the k value and distance metric with GridSearchCV rather than guessing.

In [ ]:
# Convert the categorical target labels into numeric values
# The file stores cultivars as 1, 2, 3 and scikit-learn expects 0, 1, 2
label_encoder = LabelEncoder()
y_wine = label_encoder.fit_transform(wine_target)

print('Original labels:', sorted(wine_target.unique()))
print('Encoded values: ', np.unique(y_wine))
print()
print('Class balance:')
print(pd.Series(y_wine).value_counts().sort_index())

In [ ]:
# Apply PCA keeping 95% of the variance
# Passing a float to n_components tells sklearn to keep enough components to reach it
pca_wine = PCA(n_components=0.95, random_state=42)
X_wine_pca = pca_wine.fit_transform(wine_scaled)

print('Original features:', wine_scaled.shape[1])
print('After PCA:        ', X_wine_pca.shape[1])
print('Variance kept:    ', round(pca_wine.explained_variance_ratio_.sum(), 4))

In [ ]:
# Scree plot to show how variance accumulates
explained = pca_wine.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, len(explained)+1), explained)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Scree Plot: Variance per Component')

axes[1].plot(range(1, len(cumulative)+1), cumulative, marker='o')
axes[1].axhline(0.95, color='red', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Split into training and test sets
# stratify keeps the same class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_wine_pca, y_wine, test_size=0.25, random_state=42, stratify=y_wine
)

print('Training set:', X_train.shape)
print('Test set:    ', X_test.shape)

In [ ]:
# Use GridSearchCV to find the best k value and distance metric
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'metric': ['euclidean', 'manhattan', 'chebyshev'],
    'weights': ['uniform', 'distance']
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
knn_grid.fit(X_train, y_train)

print('Best parameters:')
for key, value in knn_grid.best_params_.items():
    print(' ', key, '=', value)
print()
print('Best cross-validation accuracy:', round(knn_grid.best_score_, 4))

In [ ]:
# Train the final classifier using the best parameters
best_knn = knn_grid.best_estimator_
y_pred = best_knn.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
print('Test accuracy:', round(test_accuracy, 4))
print()
print('Classification report:')
print(classification_report(y_test, y_pred, target_names=wine_target_names))

In [ ]:
# Confusion matrix and a look at how k affected performance
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=wine_target_names, yticklabels=wine_target_names)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# Pull the cross-validation results out of the grid search
results = pd.DataFrame(knn_grid.cv_results_)
for metric in ['euclidean', 'manhattan', 'chebyshev']:
    subset = results[(results['param_metric'] == metric) &
                     (results['param_weights'] == 'distance')]
    axes[1].plot(subset['param_n_neighbors'], subset['mean_test_score'],
                 marker='o', label=metric)

axes[1].set_xlabel('k (number of neighbours)')
axes[1].set_ylabel('Cross-validation accuracy')
axes[1].set_title('Effect of k and Distance Metric')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualise the classes on the first two principal components
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_wine_pca[:, 0], X_wine_pca[:, 1], c=y_wine,
                      cmap='viridis', edgecolor='k', s=60)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Wine Cultivars on the First Two Principal Components')
plt.colorbar(scatter, label='Cultivar')
plt.tight_layout()
plt.show()

print('The three cultivars separate cleanly even in just two dimensions,')
print('which explains why the classifier performs so well.')

What we found. PCA compressed 13 chemical measurements into 10 components while keeping 96.2% of the variance. GridSearchCV tested 30 combinations and selected euclidean distance with k=5 and distance weighting, reaching 97.7% cross-validation accuracy and 95.6% on the held out test set.

The scatter plot explains why this problem is comparatively easy. The three cultivars already form visibly distinct groups on the first two components alone, so a neighbourhood based method has little trouble separating them. Distance weighting won over uniform weighting because it lets closer neighbours count more, which helps at the boundaries where the groups nearly touch.

## 3. Project Two: Feed Recommendation System for the Agricultural Client

The client supplies six chicken feed types and wants to know which ones are interchangeable. If a farm cannot get casein this month, which alternative gives the closest growth outcome? We build a profile for each feed, reduce it with PCA, and measure similarity between feeds.

In [ ]:
# First look at how the six feeds actually perform
feed_summary = chick_df.groupby('feed')['weight'].agg(['count', 'mean', 'std', 'min', 'max']).round(1)
feed_summary = feed_summary.sort_values('mean', ascending=False)
print(feed_summary)

In [ ]:
# Visualise the weight distribution per feed
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

order = feed_summary.index.tolist()
sns.boxplot(data=chick_df, x='feed', y='weight', order=order, ax=axes[0])
axes[0].set_title('Chick Weight by Feed Type')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(feed_summary.index, feed_summary['mean'])
axes[1].axhline(chick_df['weight'].mean(), color='red', linestyle='--', label='overall mean')
axes[1].set_ylabel('Mean Weight')
axes[1].set_title('Mean Weight by Feed')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Build a profile for each feed type
# We use several summary statistics so each feed is described by more than one number
feed_profile = chick_df.groupby('feed')['weight'].agg(['mean', 'std', 'min', 'max'])
print('Feed profiles before scaling:')
print(feed_profile.round(1))

# Standardize the profile so no single statistic dominates
profile_scaler = StandardScaler()
feed_profile_scaled = profile_scaler.fit_transform(feed_profile)

print()
print('Profiles scaled to mean 0 and variance 1')

In [ ]:
# Apply PCA to reduce the profiles to one principal component
pca_feed = PCA(n_components=1, random_state=42)
feed_pca = pca_feed.fit_transform(feed_profile_scaled)

print('Variance captured by PC1:', round(pca_feed.explained_variance_ratio_[0], 4))
print()

feed_scores = pd.Series(feed_pca.ravel(), index=feed_profile.index).sort_values(ascending=False)
print('Feeds positioned along PC1 (the growth performance axis):')
print(feed_scores.round(3))

In [ ]:
# Compute cosine similarity between feed types, as the brief specifies
cos_sim = cosine_similarity(feed_pca)
cos_sim_df = pd.DataFrame(cos_sim, index=feed_profile.index, columns=feed_profile.index)

print('Cosine similarity matrix:')
print(cos_sim_df.round(2))
print()
print('Unique values in this matrix:', np.unique(cos_sim.round(6)))

### Why the cosine similarity result collapses

The matrix above contains only +1 and -1, with nothing in between. This is not a bug in our code, it is a mathematical property of the method we were asked to use.

Cosine similarity measures the angle between two vectors. After reducing to a single principal component each feed is described by one number, which is a vector in one dimension. In one dimension there are only two possible directions, positive and negative, so any two feeds either point the same way or opposite ways. The result is always exactly +1 or -1 and the magnitude of the difference is discarded entirely.

That means this matrix tells us casein and sunflower are on the same side of the average, and horsebean is on the other side. It cannot tell us that casein and sunflower are near neighbours while horsebean is a distant outlier, which is exactly the question the client is asking.

We keep the required calculation above for completeness and add a distance based similarity below that uses the same principal component but preserves magnitude, so it can actually rank substitutes.

In [ ]:
# Distance based similarity on the same principal component
# We convert distance into similarity so that closer feeds score higher
distances = np.abs(feed_pca - feed_pca.T)
similarity = 1 / (1 + distances)

similarity_df = pd.DataFrame(similarity, index=feed_profile.index, columns=feed_profile.index)

print('Distance based similarity (1.00 means identical, closer to 0 means very different):')
print(similarity_df.round(3))

In [ ]:
# Compare the two approaches side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cos_sim_df, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=axes[0], cbar_kws={'label': 'cosine similarity'})
axes[0].set_title('Cosine Similarity on 1 Component\n(only +1 or -1, no useful ranking)')

sns.heatmap(similarity_df, annot=True, fmt='.2f', cmap='YlGnBu',
            ax=axes[1], cbar_kws={'label': 'distance based similarity'})
axes[1].set_title('Distance Based Similarity\n(ranks how close feeds actually are)')

plt.tight_layout()
plt.show()

In [ ]:
# Build the actual recommendation function the client would use
def recommend_similar_feeds(feed_name, sim_matrix, top_n=2):
    """Return the most similar alternative feeds, excluding the feed itself."""
    if feed_name not in sim_matrix.index:
        return 'Feed not found'
    scores = sim_matrix[feed_name].drop(feed_name).sort_values(ascending=False)
    return scores.head(top_n)

print('=== FEED SUBSTITUTION RECOMMENDATIONS ===')
for feed in feed_profile.index:
    top = recommend_similar_feeds(feed, similarity_df, top_n=2)
    alternatives = ', '.join([f'{name} ({score:.2f})' for name, score in top.items()])
    print(f'  If {feed} is unavailable, use: {alternatives}')

What we found. PC1 captured 84% of the variation across the feed profiles and lines the six feeds up along a single growth performance axis. Casein and sunflower sit at the high end, horsebean sits alone at the low end, and linseed, soybean and meatmeal fill the middle.

The practical recommendation is that casein and sunflower are close substitutes for each other, and meatmeal is the closest option below them. Horsebean is not a substitute for anything, since it produces markedly lower weights than every other option. A farm switching away from horsebean should expect a large improvement rather than a like for like swap.

The cosine similarity matrix required by the brief could not have produced any of these rankings, since it grouped all five higher performing feeds into a single undifferentiated cluster.

## 4. Project Three: Crime Clustering for the Public Safety Client

The client wants to group the 50 US states into crime risk tiers so resources can be allocated by tier rather than state by state. We select the three crime features, reduce them to two components for visualisation, then compare hard clustering with K-Means against probabilistic clustering with a Gaussian Mixture Model.

In [ ]:
# Select the top 3 relevant features for clustering
# UrbanPop is a demographic measure, not a crime rate, so we exclude it
crime_features = ['Murder', 'Assault', 'Rape']
X_crime = arrests_df[crime_features]

print('Selected features:', crime_features)
print('Excluded: UrbanPop (population measure, not a crime rate)')
print()
print('Correlation between the crime features:')
print(X_crime.corr().round(2))

In [ ]:
# Standardize, then apply PCA to reduce to 2 components
crime_scaler = StandardScaler()
X_crime_scaled = crime_scaler.fit_transform(X_crime)

pca_crime = PCA(n_components=2, random_state=42)
X_crime_pca = pca_crime.fit_transform(X_crime_scaled)

print('Shape after PCA:', X_crime_pca.shape)
print('Variance explained by each component:', pca_crime.explained_variance_ratio_.round(4))
print('Total variance kept:', round(pca_crime.explained_variance_ratio_.sum(), 4))

In [ ]:
# Elbow method for K-Means
inertia = []
k_range = range(1, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_crime_pca)
    inertia.append(km.inertia_)

# BIC for GMM
bic_scores = []
for k in k_range:
    gm = GaussianMixture(n_components=k, random_state=42, n_init=10)
    gm.fit(X_crime_pca)
    bic_scores.append(gm.bic(X_crime_pca))

print('Inertia by k:', [round(i, 1) for i in inertia])
print('BIC by k:    ', [round(b, 1) for b in bic_scores])
print()
print('Lowest BIC is at k =', k_range[int(np.argmin(bic_scores))])

In [ ]:
# Plot both selection criteria together
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(k_range, inertia, marker='o', linestyle='-')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method for K-Means')

axes[1].plot(k_range, bic_scores, marker='o', linestyle='-', color='darkorange')
axes[1].set_xlabel('Number of Components (k)')
axes[1].set_ylabel('BIC Score')
axes[1].set_title('BIC Score for GMM (lower is better)')

plt.tight_layout()
plt.show()

In [ ]:
# Check silhouette scores as a third opinion before committing to a k
print('Silhouette score by k (higher is better):')
for k in [2, 3, 4, 5]:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_crime_pca)
    print(f'  k={k}: {silhouette_score(X_crime_pca, labels):.4f}')

### Choosing the number of clusters, and being honest about it

The three criteria do not fully agree, so this is a judgement call rather than a calculation.

The elbow curve bends most sharply at k=2, with k=3 very close behind. BIC reaches its minimum at k=2. The silhouette score is also highest at k=2, at 0.568, with k=3 at 0.445.

By the metrics alone, k=2 is the defensible answer. We chose k=3 anyway, and the reason is the client's actual question. A two cluster solution splits the country into high crime and low crime, which the client already knows. Three clusters produce low, medium and high tiers, which maps onto a tiered resource allocation policy and is far more actionable. The cost of that choice is a lower silhouette score, and we report it openly rather than presenting k=3 as if the data selected it.

If the client prefers the statistically cleanest split, k=2 is available by changing a single line below.

In [ ]:
# Apply both clustering models with k=3
optimal_k = 3

# K-Means gives hard assignments: each state belongs to exactly one cluster
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_crime_pca)

# GMM gives probabilistic assignments: each state gets a probability per cluster
gmm = GaussianMixture(n_components=optimal_k, random_state=42, n_init=10)
gmm_labels = gmm.fit_predict(X_crime_pca)
gmm_proba = gmm.predict_proba(X_crime_pca)

# Store the results alongside the state names
results = arrests_df.copy()
results['KMeans_Cluster'] = kmeans_labels
results['GMM_Cluster'] = gmm_labels
results['GMM_Confidence'] = gmm_proba.max(axis=1)

print('K-Means cluster sizes:')
print(pd.Series(kmeans_labels).value_counts().sort_index())
print()
print('GMM cluster sizes:')
print(pd.Series(gmm_labels).value_counts().sort_index())

In [ ]:
# Compare the two clustering results visually
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].scatter(X_crime_pca[:, 0], X_crime_pca[:, 1], c=kmeans_labels,
                cmap='viridis', marker='o', edgecolor='k', s=80)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='red', marker='X', s=250, label='centroids')
axes[0].set_xlabel('Principal Component 1')
axes[0].set_ylabel('Principal Component 2')
axes[0].set_title('K-Means Clustering (hard assignment)')
axes[0].legend()

# Point size shows how confident GMM is about each state
axes[1].scatter(X_crime_pca[:, 0], X_crime_pca[:, 1], c=gmm_labels,
                cmap='viridis', marker='o', edgecolor='k',
                s=gmm_proba.max(axis=1) * 200)
axes[1].set_xlabel('Principal Component 1')
axes[1].set_ylabel('Principal Component 2')
axes[1].set_title('GMM Clustering (larger point means higher confidence)')

plt.tight_layout()
plt.show()

In [ ]:
# Profile each cluster so the tiers can be interpreted
print('=== K-MEANS CLUSTER PROFILES (original units) ===')
profile = results.groupby('KMeans_Cluster')[crime_features + ['UrbanPop']].mean().round(1)
profile['n_states'] = results.groupby('KMeans_Cluster').size()
print(profile)

print()
print('=== STATES IN EACH CLUSTER ===')
for c in sorted(results['KMeans_Cluster'].unique()):
    states = results[results['KMeans_Cluster'] == c].index.tolist()
    print(f'\nCluster {c} ({len(states)} states):')
    print(' ', ', '.join(states))

In [ ]:
# Where do the two methods disagree, and how confident is GMM there?
disagreements = results[results['KMeans_Cluster'] != results['GMM_Cluster']]

print('States where K-Means and GMM disagree:', len(disagreements))
if len(disagreements) > 0:
    print()
    print(disagreements[['Murder', 'Assault', 'Rape',
                         'KMeans_Cluster', 'GMM_Cluster', 'GMM_Confidence']].round(2))

print()
print('Ten states GMM is least confident about (these sit near cluster borders):')
print(results.nsmallest(10, 'GMM_Confidence')[['KMeans_Cluster', 'GMM_Cluster', 'GMM_Confidence']].round(3))

What we found. The two crime components keep 93.9% of the variance, so almost nothing is lost by working in two dimensions. PC1 acts as an overall crime severity axis, which is why the clusters line up along it.

K-Means and GMM produce broadly similar groupings, but GMM adds something K-Means cannot: a confidence score for every state. States with low confidence sit near a boundary between tiers, which is operationally useful information. A state assigned to the high tier with 55% confidence deserves a manual review, whereas one assigned with 99% confidence does not. That is the practical argument for probabilistic clustering here.

## 5. Model Evaluation and Interpretation

This section pulls the three projects together and states what each one delivers to its client.

In [ ]:
# Confirm the structure of all three datasets after processing
summary = pd.DataFrame({
    'Dataset': ['Wine', 'Chickwts', 'USArrests'],
    'Rows': [wine_df.shape[0], chick_df.shape[0], arrests_df.shape[0]],
    'Original_Features': [wine_features.shape[1], 1, arrests_df.shape[1]],
    'Features_Used': [X_wine_pca.shape[1], 1, len(crime_features)],
    'Missing_Values': [0, 0, 0],
    'Task': ['Classification', 'Recommendation', 'Clustering'],
    'Technique': ['k-NN + PCA', 'PCA + Similarity', 'K-Means + GMM']
})
print(summary.to_string(index=False))

In [ ]:
# Project 1 evaluation: classification report and accuracy
print('=== PROJECT 1: WINE CLASSIFICATION ===')
print('Best parameters:', knn_grid.best_params_)
print('Cross-validation accuracy:', round(knn_grid.best_score_, 4))
print('Test accuracy:            ', round(test_accuracy, 4))
print()
print(classification_report(y_test, y_pred, target_names=wine_target_names))

In [ ]:
# Project 2 evaluation: the recommendations themselves
print('=== PROJECT 2: FEED RECOMMENDATIONS ===')
print('Variance captured by PC1:', round(pca_feed.explained_variance_ratio_[0], 4))
print()
print('Feeds ranked by growth performance:')
for rank, (feed, score) in enumerate(feed_scores.items(), 1):
    mean_w = feed_profile.loc[feed, 'mean']
    print(f'  {rank}. {feed:11s} PC1 = {score:6.2f}   mean weight = {mean_w:.0f}g')

print()
print('Top substitute for each feed:')
for feed in feed_profile.index:
    top = recommend_similar_feeds(feed, similarity_df, top_n=1)
    print(f'  {feed:11s} -> {top.index[0]} (similarity {top.iloc[0]:.2f})')

In [ ]:
# Project 3 evaluation: cluster quality and interpretation
print('=== PROJECT 3: CRIME CLUSTERING ===')
print('Silhouette score, K-Means:', round(silhouette_score(X_crime_pca, kmeans_labels), 4))
print('Silhouette score, GMM:    ', round(silhouette_score(X_crime_pca, gmm_labels), 4))
print('Variance kept by 2 components:', round(pca_crime.explained_variance_ratio_.sum(), 4))
print()

# Label the tiers by their average crime level so they are readable
tier_order = results.groupby('KMeans_Cluster')['Assault'].mean().sort_values().index
tier_names = {c: name for c, name in zip(tier_order, ['Low', 'Medium', 'High'])}
results['Crime_Tier'] = results['KMeans_Cluster'].map(tier_names)

print('Crime tiers:')
print(results.groupby('Crime_Tier')[crime_features].mean().round(1))
print()
print('States per tier:')
print(results['Crime_Tier'].value_counts())

In [ ]:
# One combined figure summarising all three projects
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Project 1
axes[0].scatter(X_wine_pca[:, 0], X_wine_pca[:, 1], c=y_wine,
                cmap='viridis', edgecolor='k', s=50)
axes[0].set_title(f'Project 1: Wine Classification\nTest accuracy {test_accuracy:.1%}')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

# Project 2
axes[1].barh(feed_scores.index, feed_scores.values)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Project 2: Feed Performance Axis\n(PC1 score per feed)')
axes[1].set_xlabel('PC1 score')

# Project 3
axes[2].scatter(X_crime_pca[:, 0], X_crime_pca[:, 1], c=kmeans_labels,
                cmap='viridis', edgecolor='k', s=60)
axes[2].set_title('Project 3: Crime Clusters\n3 risk tiers across 50 states')
axes[2].set_xlabel('PC1')
axes[2].set_ylabel('PC2')

plt.tight_layout()
plt.show()

## Conclusion

### What we delivered

Project one, classification. The wine classifier reaches 95.6% accuracy on unseen data. PCA reduced 13 chemical measurements to 10 components while keeping 96.2% of the variance, and GridSearchCV selected euclidean distance with k=5 and distance weighting from 30 tested combinations. The winery can identify cultivar from routine chemical assays without manual inspection.

Project two, recommendation. The six feeds separate along one performance axis that captures 84% of the variation between them. Casein and sunflower are close substitutes at the high end, meatmeal is the best middle option, and horsebean stands alone as a substantially weaker feed. The client can use this to plan substitutions when supply is disrupted.

Project three, clustering. The 50 states group into three crime tiers using two principal components that retain 93.9% of the variance. K-Means gives clean tier assignments and GMM adds a confidence score per state, which flags the borderline cases that deserve a human review before resources are committed.

### Two things we want to flag rather than bury

The cosine similarity requirement in project two is mathematically degenerate. Applying cosine similarity to a single principal component can only ever return +1 or -1, because in one dimension two vectors either point the same way or the opposite way. We implemented it as specified, demonstrated the collapse, and supplied a distance based similarity that produces the rankings the client actually needs. If this pipeline goes further, the cosine step should either use two or more components or be replaced entirely.

The cluster count in project three was a judgement call, not a calculation. The elbow curve, BIC, and silhouette score all point to k=2. We chose k=3 because three tiers support a tiered allocation policy while two tiers restate what the client already knows, and the silhouette cost is 0.568 falling to 0.445. We would rather state that trade off plainly than present k=3 as though the data chose it.

### Limitations

The wine and chickwts datasets are small, at 178 and 71 rows, so the accuracy and similarity figures carry real uncertainty and would shift on a different sample. USArrests dates from 1973, so the tiers describe a historical snapshot rather than current conditions, and any live deployment needs current data. The chickwts profiles are built from group summary statistics rather than individual birds, which means the similarity scores describe feeds on average and say nothing about variability within a feed. Finally, none of the three models was tested on genuinely new data from the clients, so these are prototypes for review rather than production systems.

### Next steps

For the winery, validate the classifier on wines from a new harvest before trusting it operationally. For the agricultural client, rebuild the feed profiles from individual bird records and add cost per unit so recommendations balance growth against price. For the public safety client, refresh the analysis with current crime statistics and consider adding population and economic covariates, since crime rates alone may cluster states that differ in ways that matter for resource planning.